# RefEdit-Bench → mixed-modality prompts

Uses the official `bpathir1/RefEdit-Bench` HF test split. The benchmark provides an object-instance mask and no edited target image; the adapter preserves `edit_instruction_single`, `mask`, and Easy/Hard category semantics. Each execution creates a new `run_<timestamp>` folder under `.out/annotation/refedit_bench/` (or under the configured `ANNOTATE_OUT` root), keeping accepted annotations and audit records separate between runs. Set `RESUME_LAST_RUN = True` in the run cell to continue the latest run, skipping completed samples and retrying audited failures. `MASK_FROM_START` in the setup cell overrides `REFEDIT_MASK_FROM_START` from `.env`; when false, parsing starts text-only, but the normal conditional mask-guided retry still applies when text parsing is ambiguous or returns no usable reference.

Install once from the repository root: `uv sync`. Copy `.env.example` to `.env` and set `GEMINI_API_KEY`. The first Grounding-DINO fallback call downloads model weights; mask-only target rows do not need the fallback model.

In [1]:
from dataclasses import replace

from annotation import AnnotationConfig, refedit_bench, run_hf_annotation

DATASET_ID = "bpathir1/RefEdit-Bench"
SPLIT = "test"
CONFIG = AnnotationConfig.from_env("refedit_bench")
MASK_FROM_START = False  # Overrides REFEDIT_MASK_FROM_START from .env.
CONFIG = replace(CONFIG, refedit_mask_from_start=MASK_FROM_START)
print("Use mask from first LLM parse:", MASK_FROM_START)

Use mask from first LLM parse: False


In [2]:
from dataclasses import replace

from annotation import create_annotation_run_directory, resolve_annotation_run_directory

RESUME_LAST_RUN = True  # True continues the latest run and skips completed samples.
RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(CONFIG, output_dir=RUN_DIR)
print("Annotation run directory:", RUN_DIR)

summary = run_hf_annotation(
    dataset_id=DATASET_ID,
    split=SPLIT,
    adapter=refedit_bench,
    config=RUN_CONFIG,
    output_name="refedit_bench_mixed.jsonl",
    audit_name="refedit_bench_audit.jsonl",
    expected_columns={"input_image", "editing_instruction", "edit_instruction_single", "mask", "category"},
    use_grounder=True,
)
summary

Annotation run directory: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/refedit_bench/run_20261012_012511_868075


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


test: 0it [00:00, ?it/s]

{'accepted': 12, 'skipped': 0, 'failed': 0, 'resumed': 188, 'retried': 12}

`split` in accepted output is derived from `category` (`easy` / `hard`). `metadata.hf_split` remains `test`. RefEdit has `target_ref = null` because the native benchmark does not publish an edited target image.